# Glance–Focus ViT trên Google Colab
Chọn **Runtime → Change runtime type → GPU**, rồi chạy từng cell theo thứ tự.
Upload folder `colab` và folder `data` lên cùng một thư mục trên Google Drive.
Các module `.py` chạy độc lập bằng Python; notebook chỉ cấu hình đường dẫn và gọi CLI.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Cấu hình
Sửa `PROJECT_DIR` trỏ đến thư mục chứa `colab/` và `data/` trên Drive.
Mỗi lần chạy tạo thư mục output mới; checkpoint lưu trực tiếp lên Drive.
Dữ liệu vẫn là đa lớp. `--binary` chỉ ánh xạ sau khi dự đoán lớp bệnh.


In [ ]:
from pathlib import Path
from datetime import datetime
import os
import subprocess
import sys

PROJECT_DIR = Path('/content/drive/MyDrive/GlanceFocus')
CODE_DIR = PROJECT_DIR / 'colab'
DATA_DIR = PROJECT_DIR / 'data'
RUNS_DIR = PROJECT_DIR / 'runs'
FUSION = 'gate'  # global=E1, local=E2, concat=E3, gate=E4
SEED = 42
EPOCHS = 30
BATCH_SIZE = 2  # Tăng nếu GPU còn đủ bộ nhớ
HI_SIZE = 448
SELECTOR = None  # E2: đường dẫn đến best.pt của E1
RUN_DIR = RUNS_DIR / f'{FUSION}_seed{SEED}_{datetime.now():%Y%m%d_%H%M%S}'

required = ['train.py', 'evaluate.py', 'model.py', 'dataset.py', 'dataloader.py', 'processing.py', 'label_mapping.py']
for name in required:
    assert (CODE_DIR / name).is_file(), f'Thiếu {CODE_DIR / name}'
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


## Giải nén data.zip
Upload sẵn `data.zip` vào `/content`, rồi chạy cell bên dưới.


In [ ]:
import zipfile

with zipfile.ZipFile('/content/data.zip') as archive:
    archive.extractall('/content')

print('Đã giải nén /content/data.zip vào /content')


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('Hãy chọn GPU runtime trước khi train ViT.')
# Kiểm tra dữ liệu trong process riêng để tránh cache import của notebook.
subprocess.run([sys.executable, '-c', """
import sys
from pathlib import Path
from dataset import ImageManifest
root = Path(sys.argv[1])
train = ImageManifest(root / 'Datos/train.csv')
valid = ImageManifest(root / 'Datos/valid.csv', classes=train.classes)
assert not ({p for p, _ in train.samples} & {p for p, _ in valid.samples})
print('Classes:', train.classes)
print('Train:', len(train), 'Valid:', len(valid))
""", str(DATA_DIR)], check=True)


## Huấn luyện
Lần đầu cần Internet để tải trọng số pretrained. Train chỉ đọc train/valid.
Chạy cell cấu hình trước để thêm `colab/` vào `sys.path`; khi chạy từ notebook,
nên gọi script bằng cell lệnh bên dưới thay vì dán nội dung `train.py` vào cell.
Để chạy E2, huấn luyện E1 trước rồi đặt `SELECTOR` và đổi `FUSION='local'`.


In [ ]:
command = [sys.executable, '-u', 'train.py',
    '--train-csv', str(DATA_DIR / 'Datos/train.csv'),
    '--val-csv', str(DATA_DIR / 'Datos/valid.csv'),
    '--fusion', FUSION, '--output', str(RUN_DIR),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--hi-size', str(HI_SIZE), '--seed', str(SEED),
    '--workers', '0', '--device', 'cuda']
if SELECTOR is not None:
    command += ['--selector', str(SELECTOR)]
if FUSION == 'local' and SELECTOR is None:
    raise ValueError('E2 cần SELECTOR là checkpoint E1 đóng băng.')
subprocess.run(command, check=True)
print('Best checkpoint:', RUN_DIR / 'best.pt')


## Đánh giá external — chỉ chạy khi đã chốt cấu hình
Dùng `eardrum/test_external.csv` đã lọc và chuẩn hóa nhãn theo tập train.
Không dùng `Datos/test.csv` nếu file đó vẫn chứa những lớp bệnh ngoài tập train.
Nếu manifest nằm ở vị trí khác, sửa `TEST_CSV`. Đường dẫn ảnh tương đối được
đọc từ thư mục chứa CSV; giữ cấu trúc folder khi upload.


In [ ]:
CHECKPOINT = RUN_DIR / 'best.pt'  # Có thể thay bằng checkpoint đã lưu trên Drive
TEST_CSV = DATA_DIR / 'eardrum/test_external.csv'
RESULT_DIR = RUN_DIR / 'external_results'
command = [sys.executable, '-u', 'evaluate.py',
    '--checkpoint', str(CHECKPOINT), '--csv', str(TEST_CSV),
    '--output', str(RESULT_DIR), '--batch-size', str(BATCH_SIZE),
    '--workers', '0', '--device', 'cuda', '--binary']
if SELECTOR is not None:
    command += ['--selector', str(SELECTOR)]
subprocess.run(command, check=True)


In [ ]:
import json
report = json.loads((RESULT_DIR / 'metrics.json').read_text())
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Dự đoán:', RESULT_DIR / 'predictions.json')
